# 🏦 Banking AI-Agent: Premium Colab Edition

This notebook provides a professional, stable environment for running the **Banking AI-Agentic Workflow**. 

### ⚡ Features:
- **Llama-3 Intent Model**: Powered by your fine-tuned weights.
- **Fast Inference**: Uses Llama-3 8B for response drafting (Optimized for T4 GPU).
- **Secure Tunnels**: Powered by Cloudflare for a stable web experience.

### 🛠️ Phase 1: Environment Preparation

In [ ]:
import os
import subprocess

print("🔍 Checking GPU...")
!nvidia-smi -L

action = input("Enter Git Action (clone/pull/skip) [default: pull]: ").strip().lower() or 'pull'

if action == 'clone':
    if not os.path.exists('NLP-Project2'):
        !git clone https://github.com/TheWallOnFire/NLP-Project2.git
    %cd NLP-Project2/Ex3
elif action == 'pull':
    if os.path.exists('Ex3'):
        %cd Ex3
        !git pull
    elif os.path.exists('../.git'):
        !git pull
    else:
        print("Not in a git repo. Please clone first.")

print(f"📁 Current Directory: {os.getcwd()}")
!pip install -q -r requirements.txt
print("✅ Dependencies installed.")

### 🧠 Phase 2: Intent Model Setup
Upload the `intent_model.zip` file you downloaded from Exercise 2.

In [ ]:
from google.colab import files
import zipfile
import shutil

MODEL_DIR = "models/intent_model"
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.listdir(MODEL_DIR):
    print("📤 Please upload intent_model.zip:")
    uploaded = files.upload()
    if uploaded:
        zip_name = list(uploaded.keys())[0]
        with zipfile.ZipFile(zip_name, 'r') as zip_ref:
            zip_ref.extractall(MODEL_DIR)
        os.remove(zip_name)
        print("✅ Model extracted successfully.")
else:
    print("✅ Intent model already present. Skipping upload.")

### 💬 Phase 3: LLM Engine (Ollama)
This installs Ollama and pulls the **Llama-3 8B** model for response drafting.

In [ ]:
print("📥 Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

import threading
import time

def run_ollama():
    os.environ["OLLAMA_HOST"] = "0.0.0.0"
    subprocess.Popen(["ollama", "serve"])

threading.Thread(target=run_ollama, daemon=True).start()
time.sleep(5)

print("📥 Pulling Llama-3 8B (Fast In-Process)...")
!ollama pull llama3:8b
print("✅ Ollama is ready.")

### 🌐 Phase 4: Secure Public Access
Run this cell to generate your public web links.

In [ ]:
import re

# 1. Install Cloudflared
if subprocess.run(["which", "cloudflared"], capture_output=True).returncode != 0:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

!pkill cloudflared
time.sleep(2)

def start_tunnel(port, label):
    def log_output(process, label):
        for line in iter(process.stdout.readline, b''):
            line = line.decode('utf-8')
            if 'trycloudflare.com' in line:
                url = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
                if url: print(f"\n✅ {label} is live at: {url.group(0)}")

    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://localhost:{port}"], 
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    threading.Thread(target=log_output, args=(p, label), daemon=True).start()

print("🚀 Starting Tunnels...")
start_tunnel(8000, "BACKEND API")
start_tunnel(8501, "FRONTEND UI")
time.sleep(10)

### 🚀 Phase 5: Launch Application
The backend logs will appear below. Once you see **"Application startup complete"**, open the **FRONTEND UI** link from Phase 4.

In [ ]:
import sys

print("🚀 Starting Backend...")
def run_backend():
    env = os.environ.copy()
    env['DEBUG'] = 'true'
    env['OLLAMA_MODEL'] = 'llama3:8b'
    with subprocess.Popen([sys.executable, "run.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env) as p:
        for line in iter(p.stdout.readline, b''):
            print(f"[BACKEND] {line.decode('utf-8').strip()}")

threading.Thread(target=run_backend, daemon=True).start()
time.sleep(5)

print("🖥️ Starting Frontend UI...")
!{sys.executable} -m streamlit run frontend/app.py --server.port 8501 --server.enableCORS false